In [1]:
import ipywidgets as widgets

widgets.IntSlider(value=50, min=0, max=100, description='Value:')

IntSlider(value=50, description='Value:')

In [1]:
import ipywidgets as widgets

widgets.IntSlider(value=50, min=0, max=100, description='Value:')

IntSlider(value=50, description='Value:')

In [ ]:
import os
import rasterio

def clip_and_save_raster(src_path, dst_path):
    """
    Clips raster to largest dimensions divisible by 32 that fit within source image
    
    Args:
        src_path: Path to source TIFF file
        dst_path: Path to destination TIFF file
    """
    if os.path.exists(dst_path):
        return
        
    with rasterio.open(src_path) as src:
        data = src.read(1)
        profile = src.profile.copy()
        
        height, width = data.shape
        # Calculate largest dimensions divisible by 32
        new_height = (height // 32) * 32  # For 672 -> 672
        new_width = (width // 32) * 32    # For 576 -> 576
        

        start_y = (height - new_height) // 2
        start_x = (width - new_width) // 2
        
        clipped_data = data[start_y:start_y + new_height, 
                           start_x:start_x + new_width]
        
        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': rasterio.windows.transform(
                rasterio.windows.Window(start_x, start_y, new_width, new_height),
                src.transform
            )
        })

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        
        with rasterio.open(dst_path, 'w', **profile) as dst:
            dst.write(clipped_data, 1)

clip_and_save_raster('./Albedo.tif', './Albedo_clipped.tif')